# Stage 4 — Baseline + Merge + Experiments


## 1. Load data

In [6]:

import pandas as pd
from pathlib import Path

ROOT = Path("../")

emotion_path = ROOT / "datasets/emotion_annotated/metadata/pilot_video_emotion_features.csv"
manifest_path = ROOT / "exports/pilot/csv/pilot_manifest_export.csv"
detector_path = ROOT / "datasets/detector_processed/pilot_detector_scores.csv"

emotion_df = pd.read_csv(emotion_path)
manifest_df = pd.read_csv(manifest_path)
detector_df = pd.read_csv(detector_path)

print(len(emotion_df), len(manifest_df), len(detector_df))


200 200 200


## 2. Merge

In [13]:

# Only carry needed columns from detector_df to avoid duplicate label/split columns
detector_cols = ["video_id", "detector_score", "detector_pred"]

# Drop duplicate columns from emotion_df before merging
emotion_drop = [c for c in emotion_df.columns if c in manifest_df.columns and c != "video_id"]
df = manifest_df.merge(emotion_df.drop(columns=emotion_drop), on="video_id")
df = df.merge(detector_df[detector_cols], on="video_id")

# Encode label: "fake" → 1, "real" → 0
df["label"] = (df["label"].str.lower() == "fake").astype(int)

print("Shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())
df.head()


Shape: (200, 73)
Label distribution:
 label
0    100
1    100
Name: count, dtype: int64


,video_id,path,relative_path,split,label,source_subset,manipulation_family,manipulation_type,identity,source_identity,...,mean_score_relief,mean_score_sadness,mean_score_sexual_lust,mean_score_shame,mean_score_sourness,mean_score_teasing,mean_score_thankfulness_gratitude,mean_score_triumph,detector_score,detector_pred
0,YouTube-real__00246__bc0e9215,videos/real/YouTube-real/00246.mp4,YouTube-real/00246.mp4,train,0,YouTube-real,NaN,YouTube-real,NaN,NaN,...,-0.174653,0.083933,2.062548,0.043499,-0.680376,0.464667,0.715369,-1.077735,0.270464,0
1,Celeb-real__id19_0003__02a18478,videos/real/Celeb-real/id19_0003.mp4,Celeb-real/id19_0003.mp4,test,0,Celeb-real,NaN,Celeb-real,id19,id19,...,0.609876,-0.438989,1.791388,-0.528835,-0.732823,1.037598,3.180103,0.175943,0.435595,0
2,Celeb-real__id5_0001__7446e574,videos/real/Celeb-real/id5_0001.mp4,Celeb-real/id5_0001.mp4,train,0,Celeb-real,NaN,Celeb-real,id5,id5,...,1.041872,-0.458066,1.268688,-0.249847,-0.550585,0.728570,1.921226,-0.706379,0.918671,1
3,Celeb-synthesis__FaceSwap__MobileFaceSwap__id2...,videos/fake/FaceSwap/id21_id1_0005.mp4,Celeb-synthesis/FaceSwap/MobileFaceSwap/id21_i...,val,1,Celeb-synthesis,FaceSwap,MobileFaceSwap,id21,id21,...,0.713487,-0.253131,2.422626,-0.400127,-0.577885,0.742537,2.218336,0.044266,0.665731,1
4,Celeb-synthesis__FaceReenact__MCNET__id35_id32...,videos/fake/FaceReenact/id35_id32_0006.mp4,Celeb-synthesis/FaceReenact/MCNET/id35_id32_00...,train,1,Celeb-synthesis,FaceReenact,MCNET,id35,id35,...,0.762026,-0.332303,1.519827,-0.344126,-0.783765,0.264489,0.833108,-1.207484,0.998363,1


## 3. Metrics (baseline)

In [14]:

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

y_true = df["label"]
y_pred = df["detector_score"]

auc = roc_auc_score(y_true, y_pred)
acc = accuracy_score(y_true, (y_pred > 0.5).astype(int))
f1 = f1_score(y_true, (y_pred > 0.5).astype(int))

print("AUC:", auc)
print("ACC:", acc)
print("F1:", f1)


AUC: 0.5334000000000001
ACC: 0.53
F1: 0.6209677419354839


## 4. Emotion analysis

In [15]:

df["arousal_bin"] = pd.qcut(df["mean_arousal"], 3, labels=["low", "mid", "high"])

for g in df["arousal_bin"].unique():
    sub = df[df["arousal_bin"] == g]
    auc = roc_auc_score(sub["label"], sub["detector_score"])
    print(g, "AUC:", auc, "n=", len(sub))


mid AUC: 0.5849952516619183 n= 66
low AUC: 0.5342592592592592 n= 67
high AUC: 0.4376114081996435 n= 67


## 5. Fusion model

In [17]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

features = [
    "detector_score",
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate"
]

X = df[features].fillna(0)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score
print("Fusion AUC:", roc_auc_score(y_test, y_pred))


Fusion AUC: 0.565


## 6. Compare

In [18]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(df["label"], df["detector_score"])

emotion_features = df[[
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate"
]].fillna(0)

model2 = LogisticRegression(max_iter=1000)
model2.fit(emotion_features, df["label"])
emotion_pred = model2.predict_proba(emotion_features)[:, 1]

emotion_auc = roc_auc_score(df["label"], emotion_pred)

print("Baseline AUC:", baseline_auc)
print("Emotion AUC:", emotion_auc)


Baseline AUC: 0.5334000000000001
Emotion AUC: 0.6267
